# Lab 06 External V2 — Register Shared Tables

This notebook **does not rebuild Gold data**.

It registers Delta tables that were already written to the shared ADLS
`external_gold_root` by another workspace. This is the key cross-workspace
experiment in External V2.

## 1. Runtime parameters

In [ ]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.config import Lab06ExternalConfig, TABLE_NAMES, METRICS_TABLE
from src.external_tables import (
    ensure_target_schema,
    register_external_delta,
)

catalog = dbutils.widgets.get("catalog")
source_schema = dbutils.widgets.get("source_schema")
source_volume_name = dbutils.widgets.get("source_volume_name")
target_schema = dbutils.widgets.get("target_schema")
external_gold_root = dbutils.widgets.get("external_gold_root")
register_metrics = (
    dbutils.widgets.get("register_metrics").lower() == "true"
)

config = Lab06ExternalConfig(
    catalog=catalog,
    source_schema=source_schema,
    source_volume_name=source_volume_name,
    target_schema=target_schema,
    external_gold_root=external_gold_root,
)

print(f"Target schema : {config.target_schema_fqn}")
print(f"External root : {config.external_gold_root}")
print(f"Metrics       : {register_metrics}")

## 2. Register existing Delta paths

In [ ]:
ensure_target_schema(spark, config)

names = list(TABLE_NAMES)
if register_metrics:
    names.append(METRICS_TABLE)

results = []
failures = []

for short_name in names:
    try:
        table_name, path, row_count = register_external_delta(
            spark,
            config,
            short_name,
        )
        results.append(
            (short_name, table_name, path, row_count, "PASS", None)
        )
    except Exception as exc:
        results.append(
            (short_name, config.table(short_name),
             config.table_path(short_name), None, "FAIL", str(exc))
        )
        failures.append(short_name)

display(
    spark.createDataFrame(
        results,
        [
            "object",
            "table_name",
            "delta_path",
            "row_count",
            "status",
            "error",
        ],
    ).orderBy("object")
)

if failures:
    raise RuntimeError(
        "Registration failed for: " + ", ".join(failures)
    )

## 3. Completion

In [ ]:
print("LAB 06 EXTERNAL V2 — REGISTRATION COMPLETE")
print(f"Schema: {config.target_schema_fqn}")
print(f"Registered objects: {len(names)}")
print("No Gold transformations were rerun in this workspace.")